# 06 – LLM ocenjivač (LLM-as-a-judge) za tačnost odgovora

Ocenjuje već sačuvane odgovore (`results_generator/*/answers.jsonl`) prema činjenicama iz `eval/expected_facts.json` – **bez ponovnog generisanja**.

- Ocenjivač za svaku činjenicu kaže `present` / `missing` / `contradicted`; ocena (`correct` / `partial` / `wrong`) se izvodi pravilom u `src/judge.py`.
- Prvi prolaz: **Qwen2.5-14B**; drugi prolaz (unakrsna provera): **Gemma 4 12B** – promeni ćeliju *Podešavanja*.
- Rezultat: `judge_<JUDGE_TAG>.jsonl` za svako pokretanje + `judge_config_<JUDGE_TAG>.json`, kopirano na Drive i spakovano u zip.

Runtime: **GPU (T4)**.

## 1. Repo i paketi

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/minjaaaa/Schneider-Electric-Internship.git repo
%cd repo
!git log -1 --format="%h %ci %s"
!pip install -q -U transformers
!pip install -q python-dotenv bitsandbytes accelerate

## 2. Hugging Face token

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 3. Podešavanja
Za drugi prolaz: `JUDGE_MODEL = "google/gemma-4-12B-it"`, `MULTIMODAL = True`, `JUDGE_TAG = "gemma4-12b_j1"`.

In [ ]:
JUDGE_MODEL = "Qwen/Qwen2.5-14B-Instruct"
MULTIMODAL = False
JUDGE_TAG = "qwen14b_j1"          # ime fajla: judge_<JUDGE_TAG>.jsonl u svakom folderu pokretanja
MAX_NEW_TOKENS = 300
RUNS = [                          # prompt-v1_qwen7b se preskače: identičan je baznom pokretanju
    "2026-10-14_1022_gen_hybrid_qwen7b",
    "2026-10-14_2237_gen_hybrid_qwen14b",
    "2026-10-17_1833_gen_prompt-v2a_qwen7b",
    "2026-10-17_1837_gen_prompt-v2b_qwen7b",
    "2026-10-17_1845_gen_prompt-v2c_qwen7b",
    "2026-10-18_1708_gen_prompt-v1_gemma4-12b",
    "2026-10-18_1716_gen_prompt-v2b_gemma4-12b",
]

## 4. Činjenice i model ocenjivača

In [ ]:
import sys, json
sys.path.insert(0, "src")
from config import EVAL_DIR, GEN_RESULTS_DIR
from generate import HFChatModel
from judge import load_facts, judge_run, judge_answer, judge_fingerprint, JUDGE_VERSION

facts = load_facts(EVAL_DIR / "expected_facts.json")
missing_runs = [r for r in RUNS if not (GEN_RESULTS_DIR / r / "answers.jsonl").exists()]
assert not missing_runs, f"Nema answers.jsonl za: {missing_runs}"
print(len(facts), "pitanja,", sum(len(v) for v in facts.values()), "činjenica; prompt ocenjivača:",
      JUDGE_VERSION, judge_fingerprint())

llm = HFChatModel(JUDGE_MODEL, max_new_tokens=MAX_NEW_TOKENS, multimodal=MULTIMODAL)

## 5. Proba na jednom odgovoru
Q23 iz baznog Qwen 7B pokretanja (odgovor: 75 EUR, pogrešno) – očekuje se `wrong`. Pokreni pre punog ocenjivanja.

In [ ]:
row = next(json.loads(l) for l in open(GEN_RESULTS_DIR / RUNS[0] / "answers.jsonl", encoding="utf-8")
           if json.loads(l)["id"] == "Q23")
r = judge_answer(row, facts[row["id"]], llm)
print("ODGOVOR:", row["answer"])
print("OCENJIVAČ:", r["judge_raw"])
print("verdikti:", r["verdicts"], "| ocena:", r.get("judge_grade"), "| greška ocenjivača:", r["judge_error"],
      "|", r["judge_latency_s"], "s")

## 6. Google Drive (kopija posle svakog pokretanja)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/northstar_results/judge"
os.makedirs(DRIVE_DIR, exist_ok=True)

## 7. Ocenjivanje svih pokretanja

In [ ]:
import shutil
from collections import Counter
from tqdm.auto import tqdm

json.dump({"judge_model": JUDGE_MODEL, "judge_version": JUDGE_VERSION, "judge_hash": judge_fingerprint(),
           "facts_file": "eval/expected_facts.json", "max_new_tokens": MAX_NEW_TOKENS, "runs": RUNS},
          open(f"{DRIVE_DIR}/judge_config_{JUDGE_TAG}.json", "w"), indent=2)

for run in tqdm(RUNS):
    out = judge_run(GEN_RESULTS_DIR / run, facts, llm, JUDGE_TAG)
    shutil.copy(out, f"{DRIVE_DIR}/{run}__{out.name}")
    rows = [json.loads(l) for l in open(out, encoding="utf-8")]
    print(run, dict(Counter(r.get("judge_grade", "ERROR") for r in rows)),
          "| greške ocenjivača:", sum(r["judge_error"] for r in rows))

## 8. Preuzimanje

In [ ]:
!cd {DRIVE_DIR} && zip -q /content/judge_{JUDGE_TAG}.zip *{JUDGE_TAG}*
from google.colab import files
files.download(f"/content/judge_{JUDGE_TAG}.zip")